# 50,000 Sales Records: Profit Mix Analysis
**Data Analyst Assessment (VirtuBox) - Option A: Python**

Dataset: "50000 Sales Records" (Excel BI Analytics). 14 columns, 50,000 orders, Jan 2010 - Jul 2017.
**Important:** the publisher states the data is generated by random logic in VBA (synthetic).
All findings describe this dataset, not a real company.

Run top to bottom. Set `RAW` and `OUT` in the first code cell.

In [ ]:
import json, os, sys
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RAW = "50000_Sales_Records.csv"   # put the CSV next to the notebook (Colab: upload it first)
OUT = "output"
os.makedirs(OUT, exist_ok=True)
os.makedirs(f"{OUT}/charts", exist_ok=True)
sns.set_theme(style="whitegrid", font="DejaVu Sans")
PALETTE = {"rev": "#9CA3AF", "profit": "#0F766E", "bad": "#B45309", "accent": "#1D4ED8"}
R = {}  # results collected for the report/deck

## 1. Load and profile the raw data

In [ ]:
raw = pd.read_csv(RAW)
R["raw_rows"], R["raw_cols"] = raw.shape
print("Shape:", raw.shape)
print(raw.dtypes)
print("Missing cells:", int(raw.isna().sum().sum()))
print("Fully duplicated rows:", int(raw.duplicated().sum()))
print("Duplicate Order IDs:", int(raw["Order ID"].duplicated().sum()))
R["missing_cells"] = int(raw.isna().sum().sum())
R["dup_rows"] = int(raw.duplicated().sum())
R["dup_order_ids"] = int(raw["Order ID"].duplicated().sum())

## 2. Clean and standardise

In [ ]:
df = raw.copy()

# 2a. Text columns: trim whitespace, consistent casing
for c in ["Region", "Country", "Item Type", "Sales Channel", "Order Priority"]:
    df[c] = df[c].astype(str).str.strip()

# 2b. Dates arrive as text in US m/d/yyyy format -> real datetimes
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y", errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y", errors="coerce")
R["bad_dates"] = int(df[["Order Date", "Ship Date"]].isna().sum().sum())
R["ship_before_order"] = int((df["Ship Date"] < df["Order Date"]).sum())

# 2c. Reconciliation: do the money columns agree with each other?
tol = 0.02
R["rev_mismatch"] = int(((df["Units Sold"] * df["Unit Price"] - df["Total Revenue"]).abs() > tol).sum())
R["cost_mismatch"] = int(((df["Units Sold"] * df["Unit Cost"] - df["Total Cost"]).abs() > tol).sum())
R["profit_mismatch"] = int(((df["Total Revenue"] - df["Total Cost"] - df["Total Profit"]).abs() > tol).sum())
print({k: R[k] for k in ["bad_dates", "ship_before_order", "rev_mismatch", "cost_mismatch", "profit_mismatch"]})

# 2d. Readable priority labels (H/M/L/C per the publisher's note)
df["Order Priority Label"] = df["Order Priority"].map({"H": "High", "M": "Medium", "L": "Low", "C": "Critical"})

## 3. Calculated fields, categories, outlier flags

In [ ]:
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month
df["Order Quarter"] = "Q" + df["Order Date"].dt.quarter.astype(str)
df["Year-Month"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
df["Days To Ship"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Profit Margin"] = df["Total Profit"] / df["Total Revenue"]
df["Unit Profit"] = df["Unit Price"] - df["Unit Cost"]

# 2017 stops on 28 Jul, so it is a partial year. Flag it so nobody reads it as a decline.
last_date = df["Order Date"].max()
R["first_date"], R["last_date"] = str(df["Order Date"].min().date()), str(last_date.date())
df["Partial Year Flag"] = np.where(df["Order Year"] == last_date.year, "Partial (Jan-Jul)", "Full year")

df["Margin Tier"] = pd.cut(df["Profit Margin"], [0, 0.20, 0.35, 1.0],
                           labels=["Low (<20%)", "Medium (20-35%)", "High (>35%)"])
df["Order Size Band"] = pd.cut(df["Units Sold"], [0, 2500, 5000, 7500, 10000],
                               labels=["1-2,500", "2,501-5,000", "5,001-7,500", "7,501-10,000"])
df["Profit Band"] = pd.cut(df["Total Profit"], [0, 250_000, 1_000_000, np.inf],
                           labels=["Under $250K", "$250K-$1M", "Over $1M"])

# Outlier flag on revenue (IQR rule). Flagged, NOT removed: large orders are valid sales.
q1, q3 = df["Total Revenue"].quantile([.25, .75])
upper = q3 + 1.5 * (q3 - q1)
df["Revenue Outlier Flag"] = np.where(df["Total Revenue"] > upper, "High-value outlier", "Normal")
R["outlier_cut"] = float(upper)
R["outlier_n"] = int((df["Revenue Outlier Flag"] != "Normal").sum())
R["tiny_orders"] = int((df["Total Revenue"] < 1000).sum())
print("Outliers:", R["outlier_n"], "| orders under $1,000:", R["tiny_orders"])

df.to_csv(f"{OUT}/processed_data.csv", index=False)

## 4. Headline KPIs

In [ ]:
tot_rev, tot_profit = df["Total Revenue"].sum(), df["Total Profit"].sum()
R.update(orders=len(df), units=int(df["Units Sold"].sum()), revenue=float(tot_rev),
         cost=float(df["Total Cost"].sum()), profit=float(tot_profit), margin=float(tot_profit / tot_rev),
         aov=float(df["Total Revenue"].mean()), countries=int(df["Country"].nunique()))
print({k: R[k] for k in ["orders", "revenue", "profit", "margin", "aov", "countries"]})

## 5. Segment tables

In [ ]:
def seg(by):
    g = df.groupby(by, observed=True).agg(Orders=("Order ID", "count"), Units=("Units Sold", "sum"),
                                          Revenue=("Total Revenue", "sum"), Cost=("Total Cost", "sum"),
                                          Profit=("Total Profit", "sum"))
    g["Margin"] = g["Profit"] / g["Revenue"]
    g["Revenue Share"] = g["Revenue"] / g["Revenue"].sum()
    g["Profit Share"] = g["Profit"] / g["Profit"].sum()
    g["Profit per Order"] = g["Profit"] / g["Orders"]
    g["Revenue per Order"] = g["Revenue"] / g["Orders"]
    return g.sort_values("Revenue", ascending=False).reset_index()

item, region = seg("Item Type"), seg("Region")
channel, priority = seg("Sales Channel"), seg("Order Priority Label")
country = seg("Country")
year = seg("Order Year").sort_values("Order Year")
month = df.groupby("Year-Month").agg(Revenue=("Total Revenue", "sum"), Profit=("Total Profit", "sum"),
                                     Orders=("Order ID", "count")).reset_index()
month["Margin"] = month["Profit"] / month["Revenue"]
item_unit = df.groupby("Item Type").agg(Avg_Unit_Price=("Unit Price", "mean"), Avg_Unit_Cost=("Unit Cost", "mean"),
                                        Avg_Units_Per_Order=("Units Sold", "mean")).reset_index()
item_unit["Unit Margin"] = 1 - item_unit["Avg_Unit_Cost"] / item_unit["Avg_Unit_Price"]
reg_item_margin = df.pivot_table(index="Region", columns="Item Type", values="Profit Margin", aggfunc="mean")
ship = df.groupby("Order Priority Label")["Days To Ship"].agg(["mean", "median", "min", "max"]).reset_index()

full = year[year["Order Year"] < last_date.year]
R["rev_2010"], R["rev_2016"] = float(full.iloc[0]["Revenue"]), float(full.iloc[-1]["Revenue"])
R["cagr_2010_2016"] = (R["rev_2016"] / R["rev_2010"]) ** (1 / 6) - 1
R["yoy_full_years"] = full["Revenue"].pct_change().dropna().tolist()
R["naive_2017_drop"] = float(year.iloc[-1]["Revenue"] / year.iloc[-2]["Revenue"] - 1)
m_full = month[month["Year-Month"] < f"{last_date.year}-01-01"]
R["monthly_cv"] = float(m_full["Revenue"].std() / m_full["Revenue"].mean())
cal = df[df["Order Year"] < last_date.year].groupby("Order Month")["Total Revenue"].sum()
season = (cal / cal.mean())
R["season_min"], R["season_max"] = float(season.min()), float(season.max())

sort_profit = df["Total Profit"].sort_values(ascending=False)
R["top20pct_orders_profit_share"] = float(sort_profit.head(len(df) // 5).sum() / sort_profit.sum())
big = df[df["Total Profit"] > 1_000_000]
R["big_orders_share_of_orders"] = float(len(big) / len(df))
R["big_orders_share_of_profit"] = float(big["Total Profit"].sum() / tot_profit)
R["big_orders_by_item"] = big["Item Type"].value_counts().to_dict()
R["country_top10_rev_share"] = float(country.head(10)["Revenue"].sum() / tot_rev)

## 6. Hypothesis tests

In [ ]:
def eta_sq(col, by):
    g = df.groupby(by)[col]
    return float(((g.transform("mean") - df[col].mean()) ** 2).sum() / ((df[col] - df[col].mean()) ** 2).sum())

on, off = df[df["Sales Channel"] == "Online"], df[df["Sales Channel"] == "Offline"]
T = {}
T["channel_revenue_per_order"] = stats.ttest_ind(on["Total Revenue"], off["Total Revenue"], equal_var=False).pvalue
T["channel_margin"] = stats.ttest_ind(on["Profit Margin"], off["Profit Margin"], equal_var=False).pvalue
T["region_revenue_per_order"] = stats.f_oneway(*[g["Total Revenue"].values for _, g in df.groupby("Region")]).pvalue
T["region_margin"] = stats.f_oneway(*[g["Profit Margin"].values for _, g in df.groupby("Region")]).pvalue
T["priority_days_to_ship"] = stats.f_oneway(*[g["Days To Ship"].values for _, g in df.groupby("Order Priority")]).pvalue
T["item_profit_per_order"] = stats.f_oneway(*[g["Total Profit"].values for _, g in df.groupby("Item Type")]).pvalue
T = {k: float(v) for k, v in T.items()}
E = {f"{c}|{b}": eta_sq(c, b) for c in ["Profit Margin", "Total Profit"]
     for b in ["Item Type", "Region", "Sales Channel", "Order Priority"]}
R["pvalues"], R["eta_sq"] = T, E
R["units_uniform_ks_p"] = float(stats.kstest(df["Units Sold"] / 10_000, "uniform").pvalue)
R["item_mix_chi2_p"] = float(stats.chisquare(df["Item Type"].value_counts()).pvalue)
R["margin_by_region_min_max"] = [float(region["Margin"].min()), float(region["Margin"].max())]
print(json.dumps({"pvalues": T, "eta_sq": E}, indent=1))

## 7. What-if levers (illustrative, upper-bound arithmetic; demand response is NOT in the data)

In [ ]:
iv = item.set_index("Item Type")
S = {}
shift_meat = 0.10 * iv.loc["Meat", "Revenue"] * (iv.loc["Cosmetics", "Margin"] - iv.loc["Meat", "Margin"])
shift_off = 0.10 * iv.loc["Office Supplies", "Revenue"] * (iv.loc["Clothes", "Margin"] - iv.loc["Office Supplies", "Margin"])
S["shift10_meat_to_cosmetics"] = float(shift_meat)
S["shift10_office_to_clothes"] = float(shift_off)
for p in (0.02, 0.03, 0.05):
    S[f"price_{int(p*100)}pct_meat_office"] = float(p * (iv.loc["Meat", "Revenue"] + iv.loc["Office Supplies", "Revenue"]))
S["price_5pct_meat"] = float(0.05 * iv.loc["Meat", "Revenue"])
S["price_5pct_office"] = float(0.05 * iv.loc["Office Supplies", "Revenue"])
# volume loss that wipes out a +5% price rise (revenue constant elasticity-free arithmetic)
for it in ("Meat", "Office Supplies"):
    m = iv.loc[it, "Margin"]
    S[f"breakeven_loss_{it}"] = float(1 - m / (m + 0.05))
R["scenarios"] = S
print(S)

## 8. Charts

In [ ]:
def save(name):
    plt.tight_layout(); plt.savefig(f"{OUT}/charts/{name}.png", dpi=180); plt.show()

# 8a. revenue share vs profit share
d = item.sort_values("Revenue Share", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5.2)); y = np.arange(len(d))
ax.barh(y + .2, d["Revenue Share"] * 100, .38, color=PALETTE["rev"], label="Share of revenue")
ax.barh(y - .2, d["Profit Share"] * 100, .38, color=PALETTE["profit"], label="Share of profit")
ax.set_yticks(y); ax.set_yticklabels(d["Item Type"]); ax.set_xlabel("% of company total")
ax.set_title("Revenue share vs profit share by product category", loc="left", fontweight="bold")
ax.legend(frameon=False, loc="lower right"); sns.despine(); save("01_revenue_vs_profit_share")

# 8b. margin by category
d = item.sort_values("Margin")
fig, ax = plt.subplots(figsize=(9, 5.2))
cols = [PALETTE["bad"] if m < 0.25 else PALETTE["profit"] for m in d["Margin"]]
ax.barh(d["Item Type"], d["Margin"] * 100, color=cols)
ax.axvline(R["margin"] * 100, color="#111", ls="--", lw=1); ax.text(R["margin"] * 100 + .5, 0.2, "company avg 29.5%", fontsize=9)
for i, m in enumerate(d["Margin"]): ax.text(m * 100 + .6, i, f"{m*100:.1f}%", va="center", fontsize=9)
ax.set_xlabel("Profit margin (%)"); ax.set_title("Profit margin by category", loc="left", fontweight="bold")
sns.despine(); save("02_margin_by_category")

# 8c. annual revenue (full years) with 2017 partial
fig, ax = plt.subplots(figsize=(9, 4.6))
colors = [PALETTE["profit"] if y < last_date.year else "#CBD5E1" for y in year["Order Year"]]
ax.bar(year["Order Year"].astype(str), year["Revenue"] / 1e9, color=colors)
for x, v in zip(year["Order Year"].astype(str), year["Revenue"] / 1e9): ax.text(x, v + .1, f"${v:.1f}B", ha="center", fontsize=9)
ax.set_ylabel("Revenue ($B)"); ax.set_title("Annual revenue: flat for 7 years (2017 = Jan-Jul only)", loc="left", fontweight="bold")
sns.despine(); save("03_annual_revenue")

# 8d. days to ship by priority
fig, ax = plt.subplots(figsize=(7.5, 4.4))
order = ["Critical", "High", "Medium", "Low"]
vals = ship.set_index("Order Priority Label").loc[order, "mean"]
ax.bar(order, vals, color=PALETTE["accent"])
for i, v in enumerate(vals): ax.text(i, v + .3, f"{v:.1f} days", ha="center", fontsize=10)
ax.set_ylim(0, 30); ax.set_ylabel("Average days from order to ship")
ax.set_title("Critical orders ship no faster than Low", loc="left", fontweight="bold"); sns.despine(); save("04_ship_days_by_priority")

# 8e. region revenue + margin
d = region.sort_values("Revenue", ascending=True)
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(d["Region"], d["Revenue"] / 1e9, color=PALETTE["rev"])
for i, (r, m) in enumerate(zip(d["Revenue"] / 1e9, d["Margin"])): ax.text(r + .2, i, f"${r:.1f}B  |  margin {m*100:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, 27); ax.set_xlabel("Revenue ($B)"); ax.set_title("Regions differ in size, not in margin", loc="left", fontweight="bold")
sns.despine(); save("05_region_revenue_margin")

# 8f. monthly trend
fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.plot(month["Year-Month"], month["Revenue"] / 1e6, color=PALETTE["profit"], lw=1.6)
ax.set_ylabel("Monthly revenue ($M)"); ax.set_title("Monthly revenue, Jan 2010 - Jul 2017", loc="left", fontweight="bold"); sns.despine(); save("06_monthly_revenue")

## 9. Export tables for the workbook

In [ ]:
tables = dict(item=item, region=region, channel=channel, priority=priority, country=country, year=year,
              month=month, item_unit=item_unit, ship=ship, reg_item_margin=reg_item_margin.reset_index())
with pd.ExcelWriter(f"{OUT}/tables.xlsx") as xw:
    for k, v in tables.items(): v.to_excel(xw, sheet_name=k[:31], index=False)
for k, v in tables.items(): v.to_pickle(f"{OUT}/{k}.pkl")
json.dump(R, open(f"{OUT}/results.json", "w"), indent=1, default=str)
print("Done. Rows processed:", len(df))